In [2]:
import duckdb
import pandas as pd 
from sklearn.metrics import mean_absolute_error
con = duckdb.connect("../data/power.duckdb", read_only=True)
con.sql("SET Timezone = 'UTC'")
df = con.sql("Select * FROM features ORDER BY ts_utc").df()
con.close()

df.shape

(70801, 17)

In [4]:
train = df[df["year"].between(2019,2023)]
valid = df[df["year"] == 2024]
test = df[df["year"] == 2025]
print("train:", len(train), "hours")
print("valid:", len(valid), "hours")
print("test:", len(test), "hours")

train: 43824 hours
valid: 8784 hours
test: 8760 hours


In [ ]:
def evaluate(name, actual, predicted, threshold=0):
    """Scoring one models predicition - avg errir and negative hours that are caught"""
    actual_neg = actual < 0
    predicted_neg = predicted < threshold 
    caught = (actual_neg & predicted_neg).sum()
    false_alarms =(~actual_neg & predicted_neg).sum()
    return {
        "model": name,
        "mae_eur_mwh": round(mean_absolute_error(actual, predicted), 1),
        "negative_hours": int(actual_neg.sum()),
        "caught_pct": round(100 * caught / actual_neg.sum(), 1),
        "false_alarms_pct": round(100 * false_alarms / max(predicted_neg.sum(), 1), 1),
        
    }

What it measures:

- mae_eur_mwh: the average error. If the real prices were 50, 80, 100 and the guesses 60, 70, 100, the errors are 10, 10 and 0, so MAE = 6.7 €/MWh. Lower is better.
- caught_pct: of the hours that really were negative, how many did the model also predict as negative? ( call this recall.)
- false_alarm_pct: of the hours the model said would be negative, how many weren’t? (This is 100% minus precision.)
- & means “and”, ~ means “not”. So ~actual_neg & predicted_neg = “not actually negative, but predicted negative” = a false alarm.
- max(predicted_neg.sum(), 1) avoids dividing by zero if a model never predicts a negative hour.

In [9]:
results = pd.DataFrame([
    evaluate("same as yesterday", valid["price_euro_mwh"], valid["price_lag_24h"]),
    evaluate("same as last week", valid["price_euro_mwh"], valid["price_lag_168h"]),
])
results

,model,mae_eur_mwh,negative_hours,caught_pct,false_alarms_pct
0,same as yesterday,27.8,457,36.1,64.0
1,same as last week,32.8,457,33.5,69.2


In [10]:
check = valid.assign(
    error_yesterday=(valid["price_euro_mwh"] - valid["price_lag_24h"]).abs(),
    error_last_week=(valid["price_euro_mwh"] - valid["price_lag_168h"]).abs(),
)
check.groupby("weekday")[["error_yesterday", "error_last_week"]].mean().round(1)

,error_yesterday,error_last_week
weekday,,
1,41.1,28.6
2,24.2,30.7
3,25.9,39.0
4,25.9,38.8
5,26.2,32.3
6,27.6,28.0
7,23.7,32.3


### Baselines: two simple guessing rules (validation year 2024)

**What a baseline is:** a very simple rule that just copies an old price.
The real model must beat these rules, otherwise it's useless.

- **"Same as yesterday"** (`price_lag_24h`): to guess Tuesday 14:00,
  copy Monday 14:00.
- **"Same as last week"** (`price_lag_168h`): to guess Tuesday 14:00,
  copy last Tuesday 14:00 (7 days = 168 hours earlier).

**How we score them:**
- **MAE** (mean absolute error): how far off the guess is, on average,
  in €/MWh. Lower is better.
- **Caught %**: of the hours that really were negative, how many the rule
  also predicted as negative. Higher is better.
- **False alarm %**: of the hours the rule *said* would be negative, how
  many were not. Lower is better.

**Results for 2024:**

| Rule | MAE | Caught | False alarms |
| --- | --- | --- | --- |
| Same as yesterday | 27.8 €/MWh | 36% | 64% |
| Same as last week | 32.8 €/MWh | 34% | 69% |

**Findings:**
- "Same as yesterday" is better overall. Yesterday is only one day away,
  so the weather is usually similar (sunny or windy spells last a few days).
- **Exception: Mondays.** "Yesterday" copies Sunday, a cheap weekend day,
  so its guess for Monday (a working day) is far too low: error 41.1 €/MWh.
  "Last week" copies last Monday instead and does much better: 28.6 €/MWh.
- Both rules are weak at negative hours: they catch only about 1 in 3,
  and about 2 out of 3 of their "negative" guesses are wrong.
- The model will see both old prices **and** the weekday, so it can learn
  when to trust which one, e.g. "on Mondays, use last week".

In [13]:
FEATURES = [
    "hour","weekday","month","is_weekend","is_holiday","price_lag_24h","price_lag_48h","price_lag_168h","price_avg_7d",
    "radiation","wind_speed","temperature",
]

TARGET = "price_euro_mwh"

In [14]:
import lightgbm as lgb
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42,
    verbose=-1,
)
model.fit(train[FEATURES], train[TARGET])

,learning_rate,0.05
,n_estimators,500
,random_state,42
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [15]:
valid_pred = model.predict(valid[FEATURES])

results = pd.concat([
    results,
    pd.DataFrame([evaluate("LightGBM", valid[TARGET], valid_pred)]),
], ignore_index=True)

results

,model,mae_eur_mwh,negative_hours,caught_pct,false_alarms_pct
0,same as yesterday,27.8,457,36.1,64.0
1,same as last week,32.8,457,33.5,69.2
2,LightGBM,19.3,457,28.4,9.7


In [17]:
gain = pd.Series(
    model.booster_.feature_importance(importance_type="gain"),
    index = FEATURES,
)
(100 * gain / gain.sum()).round(1).sort_values(ascending=False)

price_lag_24h     79.6
price_lag_168h     5.3
wind_speed         5.1
weekday            2.5
price_avg_7d       2.5
price_lag_48h      2.1
temperature        0.8
radiation          0.8
hour               0.6
month              0.5
is_holiday         0.2
is_weekend         0.0
dtype: float64

### LightGBM (trained on 2019–2023, scored on 2024)

**Result:**

| Model | MAE | Caught | False alarms |
| --- | --- | --- | --- |
| Same as yesterday | 27.8 €/MWh | 36% | 64% |
| LightGBM | 19.3 €/MWh | 28% | 10% |

- Average error 31% lower than the best simple rule (19.3 vs 27.8 €/MWh).
- A careful model: when it predicts a negative hour, it is right 9 times
  out of 10. But it misses about 7 in 10 negative hours, because it was
  trained to keep the average error low and negative hours are rare (~5%).

**Most important clues (share of improvement):**
- Yesterday's price at the same hour: 80%.
- Last week's price: 5%. Wind speed: 5%.
- Sun (radiation) under 1%: yesterday's price already contains most of the
  sun pattern, while wind changes more from day to day.
- Note: importance shows what the model used, not what causes prices.

In [19]:
actual_neg = valid[TARGET] < 0
rows = []
for threshold in[0, 10, 20, 30, 40]:
    flagged = valid_pred < threshold 
    caught = (actual_neg & flagged).sum()
    false_alarms = (~actual_neg & flagged).sum()
    rows.append({
        "threshold_eur_mwh": threshold,
        "hours_flagged": int(flagged.sum()),
        "caught_pct": round(100 * caught / actual_neg.sum(), 1),
        "false_alarm_pct": round(100 * false_alarms / max(flagged.sum(), 1), 1),
    })
pd.DataFrame(rows)

,threshold_eur_mwh,hours_flagged,caught_pct,false_alarm_pct
0,0,144,28.4,9.7
1,10,264,42.2,26.9
2,20,429,60.8,35.2
3,30,676,71.8,51.5
4,40,1036,81.8,63.9


### Catching more negative hours: choosing a threshold
Instead of "predicted price below 0", we warn when the predicted price is
below a threshold. Higher threshold = more negative hours caught, but more
false alarms.

| Threshold | Hours warned | Caught | False alarms |
| --- | --- | --- | --- |
| 0 €/MWh | 144 | 28% | 10% |
| 20 €/MWh | 429 | 61% | 35% |
| 40 €/MWh | 1,036 | 82% | 64% |

- At 20 €/MWh, LightGBM beats "same as yesterday" on both measures
  (61% vs 36% caught, 35% vs 64% false alarms).
- Chosen threshold: 20 €/MWh. For charging, a false alarm is usually
  still a cheap hour, while a miss loses the cheapest power of the day.
  - The 151 false alarms at 20 €/MWh were still very cheap hours: median real
  price 2.24 €/MWh, 75% under 13 €/MWh (2024 average: __ €/MWh).
  For charging, these "mistakes" were nearly as good as hits.

In [20]:
flagged_20 = valid_pred < 20
false_alarm_prices = valid.loc[flagged_20 & ~actual_neg, TARGET]
false_alarm_prices.describe()

count    151.000000
mean      10.536755
std       16.414850
min        0.000000
25%        0.055000
50%        2.240000
75%       12.930000
max       70.470000
Name: price_euro_mwh, dtype: float64

In [21]:
valid[TARGET].mean()

np.float64(78.51203324225865)

In [22]:
tuning = []
for num_leaves in [15, 31, 63]:
    for min_child_samples in [20, 100, 300]:
        candidate = lgb.LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=num_leaves,
            min_child_samples=min_child_samples,
            random_state=42,
            verbose=-1,
        )
        candidate.fit(train[FEATURES], train[TARGET])
        mae = mean_absolute_error(valid[TARGET], candidate.predict(valid[FEATURES]))
        tuning.append({
            "num_leaves": num_leaves,
            "min_child_samples": min_child_samples,
            "mae_eur_mwh": round(mae, 2),
        })

tuning = pd.DataFrame(tuning).sort_values("mae_eur_mwh")
tuning

,num_leaves,min_child_samples,mae_eur_mwh
6,63,20,19.10
7,63,100,19.14
1,15,100,19.24
3,31,20,19.28
5,31,300,19.28
2,15,300,19.30
0,15,20,19.32
4,31,100,19.32
8,63,300,19.33


In [23]:
best = tuning.iloc[0]
best_params = {
    "num_leaves": int(best["num_leaves"]),
    "min_child_samples": int(best["min_child_samples"]),
}
best_params

{'num_leaves': 63, 'min_child_samples': 20}